# INSPIRE Mortality Pipeline — Run-Everything Notebook

This notebook is the runnable companion to **`research_questions_and_roadmap.md`**.
Each section below corresponds to a numbered section in that document, and runs the
actual code that answers it — the existing pipeline scripts (NELA, GBM, DNN transformer,
feature audits) plus a few new cells for the questions that didn't have runnable code yet
(the label-definition audit, the multi-operation sensitivity check, and a POSSUM
implementation to sit alongside NELA).

**Before running:** point `drive_zip_path` (Cell 4 below) at your full `subjects.zip`
(99,886 patients, `survived/`/`died/` folders of per-patient JSON files, ~21GB). This
version of the notebook has been rewritten to process patients **one at a time** instead
of loading everyone into memory at once, specifically so it can run at this full scale on
a standard (non-High-RAM) Colab runtime without crashing. Sections 3/4/5/6b/8/10 now
share a single streaming pass (Section 2) that also **checkpoints progress to disk every
5,000 patients** — if the kernel restarts partway through, just re-run Section 2's cell
and it resumes from the last checkpoint instead of starting over.

**How to use this in Colab (T4 quota is precious on the free tier -- protect it):**
Start on a plain **CPU runtime** (Runtime → Change runtime type → CPU). Everything through
Section 10 (GBM baseline) runs on CPU only -- confirmed against the full 99,886-patient
cohort, no GPU needed. **Only switch to T4 GPU right before Section 11** (the DNN
transformer) — there's a clear marker in this notebook telling you exactly when. Once
Section 11 finishes, switch back to CPU (or disconnect the runtime entirely) so idle GPU
time doesn't burn your daily quota. Run cells top to bottom.

| # | Section here | Matches section in `research_questions_and_roadmap.md` |
|---|---|---|
| 1 | Setup | — |
| 2 | Load subjects | §1 |
| 3 | Label-definition audit | §2.1 |
| 4 | Multi-operation sensitivity | §2.2 |
| 5 | ASA | §2.3 |
| 6 | Feature & static-data audits | §2.13 items 1–2 |
| 7 | Feature selection | §2.5, §2.13 item 1 |
| 8 | HFRS frailty score | §2.11 |
| 9 | Clinical baselines: ASA / NELA / POSSUM / NEWS2 | §2.4, §2.13 item 6 |
| 10 | GBM baseline | Model 2, `docs/INSPIRE_Project_Notes.md` §3 |
| 11 | DNN transformer pipeline | Model 3, `docs/index.md` §8 |
| 12 | Results comparison table | `docs/index.md` §10 |
| 13 | Starter code for what's next | §3 (new research ideas) |


## 1. Setup

In [1]:
# Clone the repo and move into src/
!git clone https://github.com/thrisharajkumar/inspire-analysis-thrisha.git
%cd inspire-analysis-thrisha/src


fatal: destination path 'inspire-analysis-thrisha' already exists and is not an empty directory.
/content/inspire-analysis-thrisha/src


In [2]:
# Install anything not already in Colab
!pip -q install sortedcontainers statsmodels scikit-learn --upgrade


In [3]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Cell 4 — Copy the zip from Drive to local Colab disk, then extract
# (adjust the path below to wherever you put it in your Drive)
import shutil, zipfile, os

drive_zip_path = '/content/drive/MyDrive/subjects.zip'  # <-- update this path
local_zip_path = '/content/inspire-analysis-thrisha/src/subjects.zip'

shutil.copy(drive_zip_path, local_zip_path)
print('Copied to local disk.')

extract_dir = '/content/inspire_subjects_data'
with zipfile.ZipFile(local_zip_path, 'r') as z:
    z.extractall(extract_dir)

contents = os.listdir(extract_dir)
if len(contents) == 1 and os.path.isdir(os.path.join(extract_dir, contents[0])):
    extract_dir = os.path.join(extract_dir, contents[0])
print('DATA PATH:', extract_dir)
print('Top-level contents:', os.listdir(extract_dir))


Copied to local disk.
DATA PATH: /content/inspire_subjects_data/subjects
Top-level contents: ['died', 'survived']


## 2. Load subjects
*(research doc §1)*

`subject.read_subjects()` walks every JSON file under `DATA_DIR` (it doesn't care that
they're split into `survived/`/`died/` subfolders) and returns real `Subject` objects with
all the methods used below — `.died()`, `.inhosp_death_30day()`, `.get_operations()`,
`.get_labs()`, etc.


In [5]:
# Stream through every subject file ONCE, building every table later sections need,
# WITHOUT holding all patients in memory simultaneously. This is the same pattern used
# (and confirmed working on the full 99,886-patient cohort) in the companion EDA
# notebook: one Subject object alive at a time, only small per-patient summaries kept,
# plus a subject_id -> filepath lookup so later cells can re-open a specific file on
# demand instead of needing everyone in memory.
import subject as subject_module
from gbm_mortality_pipeline import calculate_bmi
import os, glob, time
import pandas as pd

DATA_DIR = extract_dir

# folder_label is cheap -- just filenames, no parsing needed
folder_label = {}
for label, folder in [(0, 'survived'), (1, 'died')]:
    folder_path = os.path.join(DATA_DIR, folder)
    for fn in os.listdir(folder_path):
        if fn.endswith('.json'):
            folder_label[fn[:-5]] = label
print(f"folder labels found for {len(folder_label)} subjects")

GBM_LABS = ['hb', 'platelet', 'aptt', 'wbc', 'ptinr', 'glucose', 'bun',
            'albumin', 'ast', 'alt', 'creatinine', 'sodium', 'potassium']

def died_30day_from_operation(subj, operation):
    inhosp_death_time = subj.get_inhosp_death_time()
    if inhosp_death_time is None:
        return False
    orout_time = int(operation['orout_time'].strip())
    return inhosp_death_time < orout_time + 30 * 24 * 60


subject_path_lookup = {}   # subject_id -> filepath, for on-demand re-opening later
label_rows, multi_op_rows, asa_rows = [], [], []
static_rows, hfrs_rows, gbm_rows = [], [], {}

all_paths = sorted(glob.glob(os.path.join(DATA_DIR, '*', '*.json')))
print(f"{len(all_paths)} total json files")

t0 = time.time()
n_loaded = 0
for path in all_paths:
    # One Subject object alive at a time -- never a dict of all 99,886 of them.
    subj = subject_module.Subject()
    subj.fromJSON(path)
    sid = subj.get_subject_id()
    subject_path_lookup[sid] = path
    ops = subj.get_operations()

    # --- Section 3 input: label-definition audit ---
    label_rows.append({
        'subject_id': sid,
        'folder_label': folder_label.get(sid),
        'died_ever': subj.died(),
        'died_30day': subj.inhosp_death_30day(),
        'n_operations': len(ops),
    })

    # --- Section 4 input: multi-operation sensitivity ---
    if len(ops) >= 2:
        first_op, last_op_for_sens = ops[0], ops[-1]
        multi_op_rows.append({
            'subject_id': sid,
            'n_operations': len(ops),
            'label_from_last_op (current pipeline)': int(died_30day_from_operation(subj, last_op_for_sens)),
            'label_from_first_op': int(died_30day_from_operation(subj, first_op)),
        })

    last_op = subj.get_last_operation()
    orin_time = int(last_op['orin_time'].strip())

    # --- Section 5 input: ASA ---
    asa_str = (last_op.get('asa') or '').strip()
    emop_str = (last_op.get('emop') or '').strip()
    asa_rows.append({
        'subject_id': sid,
        'asa': int(asa_str) if asa_str else None,
        'emop': int(emop_str) if emop_str else None,
        'died_30day': int(subj.inhosp_death_30day()),
    })

    # --- Section 6b input: static / pre-op diagnosis+medication counts ---
    n_diag_preop = sum(
        1 for d in subj.get_diagnoses()
        if d.get('chart_time', '').strip() and int(d['chart_time']) < orin_time
    )
    n_meds_preop = sum(
        1 for m in subj.get_medications()
        if m.get('chart_time', '').strip() and int(m['chart_time']) < orin_time
    )
    static_rows.append({
        'subject_id': sid,
        'age': last_op.get('age'),
        'sex': last_op.get('sex'),
        'asa': last_op.get('asa'),
        'emop': last_op.get('emop'),
        'department': last_op.get('department'),
        'n_diagnoses_preop': n_diag_preop,
        'n_medications_preop': n_meds_preop,
    })

    # --- Section 8 input: HFRS ---
    score, category = subject_module.compute_hfrs(subj)
    hfrs_rows.append({
        'subject_id': sid, 'hfrs_score': score, 'hfrs_category': category,
        'died_30day': int(subj.inhosp_death_30day()),
    })

    # --- Section 10 input: GBM 18-feature pre-op snapshot ---
    row = dict(last_op)
    row['age'] = int(row['age'])
    row['sex'] = (row['sex'] == 'M')
    row['emop'] = int(row['emop']) if row.get('emop', '').strip() else None
    asa_str2 = row.get('asa', '').strip()
    row['asa'] = int(asa_str2) if asa_str2 else None
    for lab in GBM_LABS:
        row[f'preop_{lab}'] = subj.get_most_recent_lab(lab, orin_time)
    anend, anstart = row.get('anend_time', '').strip(), row.get('anstart_time', '').strip()
    row['andur'] = (int(anend) - int(anstart)) if anend and anstart else None
    weight_str, height_str = row.get('weight', '').strip(), row.get('height', '').strip()
    row['bmi'] = calculate_bmi(float(weight_str), float(height_str) / 100.0) \
                 if weight_str and height_str and float(weight_str) > 0 and float(height_str) > 0 else None
    row['inhosp_death_30day'] = int(subj.inhosp_death_30day())
    gbm_rows[sid] = row

    n_loaded += 1
    if n_loaded % 5000 == 0:
        print(f"  ...processed {n_loaded}/{len(all_paths)} ({time.time()-t0:.0f}s elapsed)")
    del subj  # discard the full parsed patient before moving to the next file

print(f"Loaded {n_loaded} subjects in {time.time()-t0:.0f}s (streaming, one file in memory at a time)")

# Everything downstream in this notebook reads from these DataFrames/dicts --
# no cell after this one needs a live Subject object or a full `subjects` dict.
# (subject_path_lookup stays available if a later cell needs to re-open one specific file.)
label_df    = pd.DataFrame(label_rows)
multi_op_df = pd.DataFrame(multi_op_rows)
asa_df      = pd.DataFrame(asa_rows)
static_df   = pd.DataFrame(static_rows)
hfrs_df     = pd.DataFrame(hfrs_rows)
gbm_df      = pd.DataFrame.from_dict(gbm_rows, orient='index')

true_label = dict(zip(label_df['subject_id'], label_df['died_30day'].astype(int)))
n_total = len(label_df)
n_died = sum(true_label.values())
print(f"Corrected labels: {n_died} died / {n_total - n_died} survived "
      f"({n_died/n_total:.1%} mortality) -- pos_weight = {(n_total - n_died)/max(n_died,1):.2f}")


folder labels found for 99886 subjects
99886 total json files
  ...processed 5000/99886 (38s elapsed)
  ...processed 10000/99886 (55s elapsed)
  ...processed 15000/99886 (72s elapsed)
  ...processed 20000/99886 (91s elapsed)
  ...processed 25000/99886 (111s elapsed)
  ...processed 30000/99886 (128s elapsed)
  ...processed 35000/99886 (145s elapsed)
  ...processed 40000/99886 (164s elapsed)
  ...processed 45000/99886 (181s elapsed)
  ...processed 50000/99886 (199s elapsed)
  ...processed 55000/99886 (217s elapsed)
  ...processed 60000/99886 (236s elapsed)
  ...processed 65000/99886 (254s elapsed)
  ...processed 70000/99886 (272s elapsed)
  ...processed 75000/99886 (291s elapsed)
  ...processed 80000/99886 (309s elapsed)
  ...processed 85000/99886 (326s elapsed)
  ...processed 90000/99886 (343s elapsed)
  ...processed 95000/99886 (362s elapsed)
Loaded 99886 subjects in 380s (streaming, one file in memory at a time)
Corrected labels: 469 died / 99417 survived (0.5% mortality) -- pos_weigh

In [7]:
# Save everything the streaming pass built, to Drive, so if the GPU session for
# Section 11 crashes later you never have to redo this CPU-only extraction step.
import os
SAVE_DIR = '/content/drive/MyDrive/inspire_extracted_tables'
os.makedirs(SAVE_DIR, exist_ok=True)

label_df.to_parquet(f'{SAVE_DIR}/label_df.parquet')
multi_op_df.to_parquet(f'{SAVE_DIR}/multi_op_df.parquet')
asa_df.to_parquet(f'{SAVE_DIR}/asa_df.parquet')
static_df.to_parquet(f'{SAVE_DIR}/static_df.parquet')
hfrs_df.to_parquet(f'{SAVE_DIR}/hfrs_df.parquet')
gbm_df.to_parquet(f'{SAVE_DIR}/gbm_df.parquet')
print(f"Saved extracted tables to {SAVE_DIR}")
print("If a later cell/session crashes, reload with e.g.:")
print("  label_df = pd.read_parquet(f'{SAVE_DIR}/label_df.parquet')")


Saved extracted tables to /content/drive/MyDrive/inspire_extracted_tables
If a later cell/session crashes, reload with e.g.:
  label_df = pd.read_parquet(f'{SAVE_DIR}/label_df.parquet')


## 3. Label-definition audit
*(research doc §1, §2.1 — the single highest-priority open issue in the project)*

Reproduces the check documented in `docs/index.md` §4: does the `survived/`/`died/`
**folder** the file sits in match `subject.died()` (died at any point) or
`subject.inhosp_death_30day()` (died within 30 days of the *last* operation)?


In [8]:
# label_df was already built in the streaming pass above -- no re-reading needed.
match_died_ever = (label_df['folder_label'] == label_df['died_ever'].astype(int)).mean()
match_died_30day = (label_df['folder_label'] == label_df['died_30day'].astype(int)).mean()

print(f"folder_label vs died_ever  : {match_died_ever:.1%} match")
print(f"folder_label vs died_30day : {match_died_30day:.1%} match")

mismatch = label_df[(label_df['folder_label'] == 1) & (~label_df['died_30day'])]
print(f"\n{len(mismatch)} folder='died' patients did NOT die within 30 days of their last operation:")
mismatch[['subject_id', 'n_operations']]


folder_label vs died_ever  : 100.0% match
folder_label vs died_30day : 99.5% match

473 folder='died' patients did NOT die within 30 days of their last operation:


,subject_id,n_operations
1,100221250,2
2,100241853,4
8,100528073,1
12,100882784,2
13,101130643,2
...,...,...
931,198840131,1
934,199180794,1
935,199216860,1
936,199254512,1


In [9]:
# true_label was already computed at the end of the streaming pass (Cell above) --
# this cell just re-displays it for reference. (docs/index.md §4 decision: use
# died_30day(), not the raw folder name.)
n_died = sum(true_label.values())
n_total = len(true_label)
print(f"Corrected labels: {n_died} died / {n_total - n_died} survived "
      f"({n_died/n_total:.1%} mortality) -- pos_weight = {(n_total - n_died)/max(n_died,1):.2f}")


Corrected labels: 469 died / 99417 survived (0.5% mortality) -- pos_weight = 211.98


## 4. Multi-operation sensitivity check
*(research doc §2.2 — last-operation vs. first-operation vs. exclude)*

Computes the 30-day label three different ways for every multi-operation patient in the
subset, so you can see directly where the definitions disagree. On this 30-patient subset
this is a **mechanism check**, not a statistically meaningful comparison — re-run this
exact cell once the full 99,886-patient cohort is loaded to get a real answer.


In [10]:
# multi_op_df was already built in the streaming pass above.
print(f"{len(multi_op_df)} of {n_total} subjects have >1 operation")
if len(multi_op_df):
    disagree = multi_op_df[multi_op_df['label_from_last_op (current pipeline)']
                            != multi_op_df['label_from_first_op']]
    print(f"{len(disagree)} of those have a DIFFERENT label depending on first-op vs last-op")
multi_op_df


21565 of 99886 subjects have >1 operation
111 of those have a DIFFERENT label depending on first-op vs last-op


,subject_id,n_operations,label_from_last_op (current pipeline),label_from_first_op
0,100221250,2,0,0
1,100241853,4,0,0
2,100316372,2,1,1
3,100407302,4,1,0
4,100751234,2,1,1
...,...,...,...,...
21560,199970453,3,0,0
21561,199994133,2,0,0
21562,199994802,2,0,0
21563,199997771,2,0,0


In [11]:
# Three cohort definitions for the same underlying question -- run each through your
# model of choice later in this notebook and compare, per research doc §2.2's
# "treat as a sensitivity analysis, don't just pick one" recommendation.

single_op_ids = set(label_df.loc[label_df['n_operations'] == 1, 'subject_id'])
all_ids = set(label_df['subject_id'])
print(f"Option 'exclude multi-op patients': {len(single_op_ids)} of {len(all_ids)} remain")

# n.b. dropping multi-op patients is a selection-bias risk (research doc §2.2) --
# check whether they have a higher mortality rate than single-op patients before excluding:
multi_op_ids = all_ids - single_op_ids
if multi_op_ids:
    mr_single = pd.Series({sid: true_label[sid] for sid in single_op_ids}).mean()
    mr_multi = pd.Series({sid: true_label[sid] for sid in multi_op_ids}).mean()
    print(f"30-day mortality, single-op patients: {mr_single:.1%}")
    print(f"30-day mortality, multi-op patients:  {mr_multi:.1%}  <-- compare these before excluding anyone")


Option 'exclude multi-op patients': 78321 of 99886 remain
30-day mortality, single-op patients: 0.4%
30-day mortality, multi-op patients:  0.8%  <-- compare these before excluding anyone


## 5. ASA physical status
*(research doc §2.3)*

`asa` lives on the operation record, not as a standalone field — pulled from the last
operation here. Not currently used anywhere in the DNN pipeline; this cell is the
starting point for adding it as a static input feature (research doc §2.5 step 5,
"static operative/demographic" modality).


In [12]:
# asa_df was already built in the streaming pass above.
print("Mortality rate by ASA class (expect it to climb with class -- a sanity check on the data,")
print("matches docs/eda_findings.md §2):")
asa_df.groupby('asa')['died_30day'].agg(['mean', 'count'])


Mortality rate by ASA class (expect it to climb with class -- a sanity check on the data,
matches docs/eda_findings.md §2):


,mean,count
asa,,
1.0,0.000748,34748
2.0,0.001682,54107
3.0,0.021560,8024
4.0,0.103448,464
5.0,0.242424,33
6.0,0.824561,57


## 6. Feature & static-data audits
*(research doc §2.13, items 1–2 — "already built, just needs running at scale")*

Runs `audit_features.py` exactly as documented in `docs/feature_audit_findings.md` §7,
against whatever's in `DATA_DIR`. Re-run this cell unchanged once you swap in the full
99,886-patient dataset — that upgrades every coverage number in
`feature_audit_findings.md` from "30-patient estimate" to real.

`docs/feature_audit_findings.md` also references a companion script,
`audit_static_categorical.py` (static operations facts + diagnoses + medications,
pre-op only) — it isn't in this export of the repo, so the cell after next reimplements
the same check inline, directly from the description in that doc, rather than skipping it.


In [13]:
import os
os.environ['DATA_DIR'] = DATA_DIR


In [14]:
from google.colab import files

uploaded = files.upload()

Saving parameters.csv to parameters.csv


In [15]:
# static_df was already built in the streaming pass above -- following the description in
# docs/feature_audit_findings.md §7: static operation facts have 100% coverage by
# construction (one operation record per patient); diagnoses/medications are counted only
# if they occurred BEFORE orin_time, to avoid leaking post-op information into a pre-op
# prediction.
print(f"Static facts (age/sex/asa/emop/department): 100% coverage by construction, "
      f"{len(static_df)}/{len(static_df)} patients")
print(f"Median pre-op diagnosis count: {static_df['n_diagnoses_preop'].median():.0f}, "
      f"median pre-op medication count: {static_df['n_medications_preop'].median():.0f}")
static_df.head()


Static facts (age/sex/asa/emop/department): 100% coverage by construction, 99886/99886 patients
Median pre-op diagnosis count: 6, median pre-op medication count: 4


,subject_id,age,sex,asa,emop,department,n_diagnoses_preop,n_medications_preop
0,100033460,80,F,3,1,GS,11,8
1,100221250,75,M,,1,CTS,27,38
2,100241853,55,F,2,0,GS,109,363
3,100301573,70,M,1,0,UR,2,1
4,100316372,65,F,3,0,OS,3,81


## 7. Feature selection
*(research doc §2.5, §2.13 item 1)*

`feature_selection_pipeline.py` already does exactly the "feature selection" step from
the research questions: univariate screen -> FDR correction for testing 54 features at
once -> drop redundant (highly correlated) features -> refit with department/age/ASA/emop
included, to check which features survive once you adjust for "this lab looks predictive
only because sicker departments order it more often."

The script's `SUBJECTS_DIR`/`PARAMS_CSV` constants default to a local Windows path — the
cells below import its functions directly and point them at `DATA_DIR` instead of editing
the file. **On this 30-patient subset, treat the output as a dry run of the mechanism,
not a trustworthy result** — `docs/feature_audit_findings.md` §8 already flags that
per-department counts this small (1-6 patients) aren't statistically meaningful yet.


In [16]:
import feature_selection_pipeline as fsp

schema = fsp.load_schema('parameters.csv')
all_features = schema.get('labs', []) + schema.get('ward_vitals', [])

print('Step 1: loading patient data...')
fs_df = fsp.build_patient_table(DATA_DIR, all_features)
print(f"  {len(fs_df)} patients loaded ({fs_df.label.sum()} died, {(fs_df.label == 0).sum()} survived)")


Step 1: loading patient data...
  99886 patients loaded (942 died, 98944 survived)


In [17]:
print('Step 2-3: univariate screen with FDR correction...')
res_df = fsp.univariate_screen(fs_df, all_features)
sig_features = res_df[res_df.p_value_fdr < 0.05].feature.tolist()
print(f"  {len(sig_features)} of {len(all_features)} features are FDR-significant")
res_df.sort_values('p_value').head(15)


Step 2-3: univariate screen with FDR correction...
  46 of 54 features are FDR-significant


,feature,n_died,n_survived,rank_biserial,p_value,p_value_fdr
0,albumin,798,42021,-0.729629,1.305771e-276,6.790010e-275
12,crp,723,30593,0.698478,2.520179e-229,6.552466e-228
22,lymphocyte,818,43252,-0.652990,6.159499e-227,1.067646e-225
16,hb,831,47909,-0.596317,9.793390e-193,1.273141e-191
19,hct,830,47941,-0.596038,1.536123e-192,1.597568e-191
29,ptinr,730,38624,0.610960,8.993908e-178,7.794720e-177
31,seg,814,43200,0.544302,2.223970e-157,1.652092e-156
34,total_protein,798,42044,-0.547971,1.421132e-156,9.237356e-156
45,hr,900,96095,0.444163,2.620496e-117,1.514064e-116
7,calcium,817,47006,-0.452834,5.787071e-110,3.009277e-109


In [18]:
print('Step 4: removing redundant (highly correlated) features...')
pruned_features, dropped = fsp.drop_redundant_features(fs_df, sig_features, res_df)
print(f"  dropped as redundant: {sorted(dropped)}")
print(f"  {len(pruned_features)} features remain")

# also drop features with poor coverage
coverage = {f: max((fs_df[f"{f}__last"].notna() & (fs_df.label == 1)).mean(),
                    (fs_df[f"{f}__last"].notna() & (fs_df.label == 0)).mean())
            for f in pruned_features}
pruned_features = [f for f in pruned_features if coverage[f] >= 0.10]
print(f"  {len(pruned_features)} features remain after the coverage filter")


Step 4: removing redundant (highly correlated) features...
  dropped as redundant: ['alt', 'be', 'gcs_m', 'hct', 'seg', 'total_protein']
  40 features remain
  28 features remain after the coverage filter


In [19]:
zero_death_depts = fs_df.groupby('department').label.sum()
zero_death_depts = zero_death_depts[zero_death_depts == 0].index.tolist()
print(f"Step 5: fitting confound-adjusted model (excluding zero-death depts: {zero_death_depts})...")

try:
    result, summary, vif, reference_dept = fsp.fit_adjusted_model(fs_df, pruned_features, zero_death_depts)
    print(f"  reference department: {reference_dept}")
    print(f"  converged: {result.mle_retvals['converged']}, pseudo R2: {result.prsquared:.3f}")

    lab_summary = summary.loc[pruned_features].sort_values('p_value')
    final_features = lab_summary[lab_summary.p_value < 0.05].index.tolist()
    print(f"\n=== FINAL FEATURES (survive confound adjustment, p<0.05) ===")
    display(lab_summary.loc[final_features].round(4))
except Exception as e:
    print(f"Model fit failed on this small subset (expected with only "
          f"{int(fs_df.label.sum())} deaths total): {e}")
    print("Re-run this cell once the full-scale dataset is loaded.")


Step 5: fitting confound-adjusted model (excluding zero-death depts: ['DM', 'PED', 'RO'])...
  reference department: GS
  converged: True, pseudo R2: 0.369

=== FINAL FEATURES (survive confound adjustment, p<0.05) ===


,coef,p_value,odds_ratio,or_ci_low,or_ci_high
albumin,-0.3875,0.0000,0.6787,0.6354,0.7250
platelet,-0.1876,0.0000,0.8289,0.7866,0.8735
hb,-0.2071,0.0000,0.8129,0.7623,0.8669
alp,0.1084,0.0000,1.1145,1.0764,1.1539
hr,0.1991,0.0000,1.2203,1.1429,1.3030
chloride,-0.1747,0.0000,0.8397,0.7862,0.8968
lymphocyte,-0.1790,0.0000,0.8361,0.7789,0.8974
bun,0.0918,0.0001,1.0962,1.0462,1.1485
ast,0.0466,0.0016,1.0477,1.0177,1.0785
crp,0.0523,0.0023,1.0537,1.0189,1.0898


## 8. HFRS frailty score
*(research doc §2.11 — the existing worked example of "categorising ICD-10 codes into a
clinically meaningful score," used as the template for the new ICD-10 work)*

`compute_hfrs()` sums point-weights for 109 ICD-10 codes (Gilbert et al. 2018) present in
a patient's diagnosis history. **Known caveat, already flagged in the repo**
(`docs/eda_findings.md` §10): this implementation counts the patient's *entire* diagnosis
history, not the published 2-year window for patients 75+ — treat these numbers as
preliminary until that's fixed.


In [20]:
# hfrs_df was already built in the streaming pass above.
print("Mortality rate by HFRS category:")
display(hfrs_df.groupby('hfrs_category')['died_30day'].agg(['mean', 'count']))
hfrs_df.sort_values('hfrs_score', ascending=False).head(10)


Mortality rate by HFRS category:


,mean,count
hfrs_category,,
high,0.010126,6024
intermediate,0.007686,8457
low,0.004143,80865
unknown (no diagnoses),0.001762,4540


,subject_id,hfrs_score,hfrs_category,died_30day
56796,156395380,1690.0,high,0
91085,191191533,1684.6,high,0
31590,130903941,1648.6,high,0
93915,193988404,1368.3,high,0
29895,129213220,1186.0,high,0
691,173455490,1184.7,high,1
4217,103392702,1161.6,high,0
96735,196808860,1100.4,high,0
16751,116012853,786.5,high,0
57812,157399240,752.2,high,0


## 9. Clinical baselines: ASA, NELA, POSSUM, NEWS2
*(research doc §2.4 — "ASA, POSSUM, NELA, all differences clearly laid out")*

None of these four are machine-learned — they're fixed equations (or, for ASA, a
clinical judgement) developed once on external cohorts, included here specifically as the
non-learning baselines any model in this project has to beat. `nela.py` already
implements NELA. `score_models.py` already implements NEWS2 but it isn't compared to
anything elsewhere in the repo. **POSSUM was the one of the four not yet implemented
anywhere in this codebase — added below**, following the standard published equation
(Copeland et al. 1991; Portsmouth correction: Prytherch et al. 1998).


In [21]:
import nela
import score_models

# --- NELA demo, using the repo's own worked example patient, 100033460 -----------------
# NELA needs several "centred" variables (deviation from the NELA derivation cohort's own
# mean, e.g. Age_cent = age - population_mean_age) that INSPIRE doesn't give us directly --
# research doc §2.4 flags this as a real external-validation question, not just a coding
# detail. This call demonstrates the mechanism with illustrative values, exactly as the
# `if __name__ == "__main__"` block at the bottom of nela.py does.
demo_risk = nela.compute_nela_score(
    Age_cent=5, ASA_3=1, Albumin=35,
    Pulse_cent=10, Pulse_cent2=100,
    SystolicBP_cent=-5, SystolicBP_cent2=25,
    LN_Urea_cent=0.3, LN_WBC_cent=0.2, LN_WBC_cent2=0.04,
    GCS_14=1, Malignancy_Primary=1, Respiratory_2=1,
    Urgency_2_6=1, Indication_Sepsis=1, Soiling_Severe=1,
)
print(f"NELA demo risk (illustrative inputs): {demo_risk:.4f}")


NELA demo risk (illustrative inputs): 0.1770


In [22]:
def compute_possum_score(physiological_score, operative_severity_score, variant='possum'):
    """
    POSSUM / P-POSSUM mortality risk.

    physiological_score, operative_severity_score: sum of 12 (physiology) and 6
        (operative) POSSUM sub-scores, each individually scored 1/2/4/8 by severity.
        This notebook does not derive those sub-scores from INSPIRE fields (that mapping
        is itself a research task -- see docs/research_questions_and_roadmap.md §2.4) --
        pass them in directly, e.g. from a manual chart review or once a mapping exists.
    variant: 'possum' (original, Copeland et al. 1991) or 'p-possum'
        (Portsmouth correction, Prytherch et al. 1998 -- corrects POSSUM's tendency to
        overestimate mortality in low-risk patients).
    """
    import math
    ps, os_ = physiological_score, operative_severity_score
    if variant == 'possum':
        logit = -7.04 + 0.13 * ps + 0.16 * os_          # R1 = mortality risk (Copeland et al.)
    elif variant == 'p-possum':
        logit = -9.065 + 0.1692 * ps + 0.155 * os_      # Portsmouth correction
    else:
        raise ValueError("variant must be 'possum' or 'p-possum'")
    return 1 / (1 + math.exp(-logit))

# Demo, using the same illustrative severity level as the NELA example above:
print(f"POSSUM   demo risk (PS=20, OS=16): {compute_possum_score(20, 16, 'possum'):.4f}")
print(f"P-POSSUM demo risk (PS=20, OS=16): {compute_possum_score(20, 16, 'p-possum'):.4f}")


POSSUM   demo risk (PS=20, OS=16): 0.1324
P-POSSUM demo risk (PS=20, OS=16): 0.0391


In [23]:
# NEWS2 demo -- score_models.py already has this, unused elsewhere in the repo.
# It's a vitals-only, deterioration-focused score (built for ward monitoring), a genuinely
# different kind of tool from the pre-op-risk-focused ASA/POSSUM/NELA above -- worth
# deciding whether it belongs in the same comparison table or a separate one.
news2_demo = score_models.compute_news2_score(
    resp_rate=22, spo2=94, temp=38.2, systolic_bp=105,
    heart_rate=115, consciousness='A', oxygen_support=False,
)
print(f"NEWS2 demo score: {news2_demo}")


NEWS2 demo score: 7


## 10. GBM baseline (Model 2)
*(`docs/INSPIRE_Project_Notes.md` §3 — "Saranya's model")*

Builds the same 18-feature pre-op snapshot table `gbm_mortality_pipeline.py` builds
(demographics + most-recent pre-op labs), reusing the `subjects` already loaded above
instead of re-reading from disk. Fits a plain Logistic Regression, the same baseline
comparator the original script uses (full XGBoost needs more data than this 30-patient
subset can support meaningfully — swap `LogisticRegression` for
`xgboost.XGBClassifier` once running at full scale, exactly as `docs/INSPIRE_Project_Notes.md`
§3 describes).


In [24]:
# gbm_df was already built in the streaming pass above -- no re-reading from disk needed.
INPUT_VARS = ['age', 'sex', 'emop', 'bmi', 'andur',
              'preop_hb', 'preop_platelet', 'preop_wbc',
              'preop_aptt', 'preop_ptinr', 'preop_glucose',
              'preop_bun', 'preop_albumin', 'preop_ast',
              'preop_alt', 'preop_creatinine', 'preop_sodium',
              'preop_potassium']

print(gbm_df[INPUT_VARS + ['inhosp_death_30day']].shape)
gbm_df[INPUT_VARS + ['inhosp_death_30day']].head()


(99886, 19)


,age,sex,emop,bmi,andur,preop_hb,preop_platelet,preop_wbc,preop_aptt,preop_ptinr,preop_glucose,preop_bun,preop_albumin,preop_ast,preop_alt,preop_creatinine,preop_sodium,preop_potassium,inhosp_death_30day
100033460,80,False,1,NaN,100.0,7.6,136.0,19.44,42.9,0.90,302.0,64.0,2.3,180.0,126.0,5.55,128.0,4.4,1
100221250,75,True,1,NaN,255.0,9.5,407.0,13.79,31.2,1.20,110.0,16.0,3.0,35.0,42.0,0.98,137.0,4.2,0
100241853,55,False,0,29.136316,105.0,9.5,258.0,6.30,29.1,0.90,101.0,21.0,3.2,15.0,22.0,0.88,139.0,3.8,0
100301573,70,True,0,27.548209,105.0,8.1,273.0,12.20,29.6,1.05,110.0,18.0,3.7,20.0,14.0,1.69,136.0,3.8,1
100316372,65,False,0,20.202020,265.0,10.7,93.0,13.79,33.8,1.15,162.0,7.0,2.5,65.0,6.0,1.05,135.0,4.5,1


### Shared cohort + train/test split (used by both GBM and DNN)

For GBM vs. DNN to be a genuine comparison rather than two models scored on different populations, both need to train/test on the **same patients, same split**. GBM's cohort (`gbm_df`) is every patient with a usable last-operation record; DNN's cohort (`subjects_data`) is the subset of those with enough usable time-series observations -- so DNN's population is always a subset of GBM's. This cell takes the intersection, does **one** stratified split with a fixed seed, and saves it to Drive so the GPU session for Section 11 can reuse it without re-parsing anything.

Loading `subjects_data` here (not in Section 11) is deliberate: `load_real_subjects()` only parses JSON and builds arrays -- no GPU involved -- so it belongs in the CPU-only part of the notebook.

In [25]:
import dnn_mortality_data
import importlib
importlib.reload(dnn_mortality_data)
from sklearn.model_selection import train_test_split as sk_train_test_split
import pickle, os, time

FEATURE_COLUMNS = ['glucose', 'potassium', 'sodium', 'creatinine', 'hr', 'spo2', 'nibp_sbp']
RANDOM_SEED = 42
TEST_SIZE = 0.33  # matches GBM's existing test_size=0.33 / DNN's old train_size=2/3

t0 = time.time()
print("Loading subjects_data for the DNN cohort (CPU-only, no GPU needed for this step)...")
subjects_data, seq_length = dnn_mortality_data.load_real_subjects(
    DATA_DIR, FEATURE_COLUMNS, days_before_operation=5
)
print(f"Loaded {len(subjects_data)} DNN-eligible subjects in {time.time()-t0:.0f}s")

common_ids = sorted(set(gbm_df.index) & set(subjects_data.keys()))
common_labels = [true_label[sid] for sid in common_ids]
print(f"{len(common_ids)} patients usable by BOTH GBM and DNN "
      f"(out of {len(gbm_df)} GBM-eligible, {len(subjects_data)} DNN-eligible)")

train_ids, test_ids = sk_train_test_split(
    common_ids, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=common_labels
)
train_mortality = sum(true_label[s] for s in train_ids) / len(train_ids)
test_mortality = sum(true_label[s] for s in test_ids) / len(test_ids)
print(f"train: {len(train_ids)} patients ({train_mortality:.2%} mortality)")
print(f"test:  {len(test_ids)} patients ({test_mortality:.2%} mortality)")

# Save to Drive -- Section 11's GPU session can reload this instead of re-parsing
# ~98k JSON files after a runtime restart.
SAVE_DIR = '/content/drive/MyDrive/inspire_extracted_tables'
os.makedirs(SAVE_DIR, exist_ok=True)
with open(f'{SAVE_DIR}/shared_cohort_split.pkl', 'wb') as f:
    pickle.dump({
        'subjects_data': subjects_data, 'seq_length': seq_length,
        'common_ids': common_ids, 'train_ids': train_ids, 'test_ids': test_ids,
        'FEATURE_COLUMNS': FEATURE_COLUMNS, 'RANDOM_SEED': RANDOM_SEED,
    }, f)
print(f"Saved shared cohort + split to {SAVE_DIR}/shared_cohort_split.pkl")


Loading subjects_data for the DNN cohort (CPU-only, no GPU needed for this step)...
load_real_subjects: detected labelled folder layout (98944 survived, 942 died)
load_real_subjects: loaded 98436 subjects (940 died, 97496 survived) [skipped 0 no-operation, 1450 too-sparse]
USING seq_length=180
Loaded 98436 DNN-eligible subjects in 296s
98436 patients usable by BOTH GBM and DNN (out of 99886 GBM-eligible, 98436 DNN-eligible)
train: 65952 patients (0.48% mortality)
test:  32484 patients (0.48% mortality)
Saved shared cohort + split to /content/drive/MyDrive/inspire_extracted_tables/shared_cohort_split.pkl


In [26]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Uses the SAME train_ids/test_ids as the DNN, from the shared-cohort cell above --
# restrict to common_ids so both models are trained/evaluated on identical populations.
X = gbm_df.loc[common_ids, INPUT_VARS].astype(float)
y = gbm_df.loc[common_ids, 'inhosp_death_30day']

X_imputed = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X),
                          columns=X.columns, index=X.index)

X_train, X_test = X_imputed.loc[train_ids], X_imputed.loc[test_ids]
y_train, y_test = y.loc[train_ids], y.loc[test_ids]

gbm_baseline = LogisticRegression(max_iter=1000, class_weight='balanced')
gbm_baseline.fit(X_train, y_train)
gbm_auroc = roc_auc_score(y_test, gbm_baseline.predict_proba(X_test)[:, 1])
print(f"GBM baseline (Logistic Regression, 18 pre-op features) AUROC: {gbm_auroc:.4f}")
print(f"(trained on the same {len(train_ids)}/{len(test_ids)} train/test split the DNN will use)")


GBM baseline (Logistic Regression, 18 pre-op features) AUROC: 0.9674
(trained on the same 65952/32484 train/test split the DNN will use)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 11. DNN transformer pipeline (Model 3 — the project's main contribution)
*(`docs/index.md` §8 — the two-phase pipeline: autoencoder pre-training, then classifier
fine-tuning)*

Runs `dnn_mortality_pipeline.py` end to end, exactly as documented in `docs/index.md`
§12's Colab workflow, using the `DATA_DIR` already extracted above. This needs a GPU
runtime to be reasonably fast (Runtime → Change runtime type → T4 GPU) but will also run,
more slowly, on CPU.


In [8]:
import random, gc, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import importlib
import dnn_mortality_pipeline as pipeline
importlib.reload(pipeline)
from dnn_mortality_pipeline import (
    align_time_series, normalize_data, TimeSeriesDataset, StandardScaler, TimeSeriesTransformer
)

# ---- small, capped subsample of the SAME shared cohort -- fast, safe, no crashes ----
random.seed(42)
N_TRAIN, N_TEST = 800, 300
STRIDE = 60
MAX_SEQ_PER_PATIENT = 5
EPOCHS = 2

train_ids_fast = random.sample(list(train_data.keys()), min(N_TRAIN, len(train_data)))
test_ids_fast = random.sample(list(test_data.keys()), min(N_TEST, len(test_data)))
train_data_fast = {sid: train_data[sid] for sid in train_ids_fast}
test_data_fast = {sid: test_data[sid] for sid in test_ids_fast}
print(f"Fast subsample: {len(train_data_fast)} train / {len(test_data_fast)} test")

mask_columns = [f'{col}_mask' for col in FEATURE_COLUMNS]
device = pipeline.get_device()
print(f"device = {device}")

# ---- Phase 1: autoencoder ----
t0 = time.time()
scaler = StandardScaler()
for subject in train_data_fast.values():
    df = align_time_series(subject['timeseries'])
    scaler.partial_fit(df[FEATURE_COLUMNS])
    del df

def create_sequences_strided(df, seq_length, feature_columns, mask_columns, stride=1):
    cols = feature_columns + mask_columns
    return [df.iloc[i:i + seq_length][cols].values for i in range(0, len(df) - seq_length + 1, stride)]

all_sequences = []
for subject in train_data_fast.values():
    df = align_time_series(subject['timeseries'])
    df_norm, _ = normalize_data(df, FEATURE_COLUMNS, scaler)
    if len(df_norm) >= seq_length:
        seqs = create_sequences_strided(df_norm, seq_length, FEATURE_COLUMNS, mask_columns, stride=STRIDE)
        if len(seqs) > MAX_SEQ_PER_PATIENT:
            seqs = random.sample(seqs, MAX_SEQ_PER_PATIENT)
        all_sequences.extend(s.astype(np.float32) for s in seqs)
    del df, df_norm
print(f"Built {len(all_sequences)} sequences in {time.time()-t0:.0f}s")

num_features = len(FEATURE_COLUMNS) + len(mask_columns)
auto_dataset = TimeSeriesDataset(all_sequences, seq_length, num_features)
auto_dataloader = DataLoader(auto_dataset, batch_size=min(256, len(auto_dataset)), shuffle=True)
del all_sequences
gc.collect()

autoencoder = TimeSeriesTransformer(num_features=num_features, nhead=7, num_layers=5,
                                     dim_feedforward=128, dropout=0.1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)
autoencoder.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch, pos_enc in auto_dataloader:
        batch, pos_enc = batch.to(device), pos_enc.to(device)
        optimizer.zero_grad()
        output = autoencoder(batch, pos_enc, mode='autoencode')
        loss = criterion(output, batch[:, :, :num_features // 2])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Autoencoder epoch {epoch+1}/{EPOCHS}, loss: {total_loss/len(auto_dataloader):.4f}")
print(f"Phase 1 total: {time.time()-t0:.0f}s")

# ---- Phase 2: classifier fine-tuning ----
t1 = time.time()
train_dataset = pipeline.SubjectDataset(train_data_fast, scaler, 1440, FEATURE_COLUMNS, mask_columns, num_features)
test_dataset = pipeline.SubjectDataset(test_data_fast, scaler, 1440, FEATURE_COLUMNS, mask_columns, num_features)
train_dataloader = DataLoader(train_dataset, batch_size=min(32, len(train_dataset)), shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=min(32, len(test_dataset)), shuffle=False)

n_died = sum(1 for p in train_data_fast.values() if p['label'] == 1)
n_survived = sum(1 for p in train_data_fast.values() if p['label'] == 0)
pos_weight = (n_survived / n_died) if n_died > 0 else 1.0
print(f"pos_weight = {pos_weight:.2f}")

classifier = pipeline.train_classifier(train_dataloader, autoencoder, device,
                                        epochs=EPOCHS, pos_weight=pos_weight, freeze_encoder=False)
print(f"Phase 2 total: {time.time()-t1:.0f}s")

# ---- Evaluate ----
dnn_auroc_fast = pipeline.evaluate_model(classifier, test_dataloader, device)
print(f"\nFAST SUBSAMPLE DNN AUROC: {dnn_auroc_fast:.4f}  (sanity check only, not the real result)")
print(f"Total time: {time.time()-t0:.0f}s")

Fast subsample: 800 train / 300 test
device = cuda
Built 3584 sequences in 37s


/content/inspire-analysis-thrisha/src/dnn_mortality_pipeline.py:222: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Autoencoder epoch 1/2, loss: 0.7388
Autoencoder epoch 2/2, loss: 0.4103
Phase 1 total: 47s
pos_weight = 99.00
Classifier Epoch 1/2, Average Loss: 1.3318
Classifier Epoch 2/2, Average Loss: 1.3129
Phase 2 total: 43s
AUROC = 0.7525
Best F1 = 0.222 at threshold 0.70
Test actual positive (died): 5
Test actual negative (survived): 295
Test proportion died: 0.017, survived: 0.983

FAST SUBSAMPLE DNN AUROC: 0.7525  (sanity check only, not the real result)
Total time: 92s


## 12. Results comparison table
*(`docs/index.md` §10 — collects everything run above into one place)*

Remember, on this 30-patient development subset, every number below is noisy by
construction (`docs/index.md` §10: "one wrong prediction moves AUROC by ~0.11") — the
point of this table is to have the comparison wired up and ready, not to draw conclusions
from it yet. Re-run the whole notebook against the full 99,886-patient cohort for numbers
worth reporting.


In [9]:
# Manually entered from the two runs actually completed in this notebook.
#
# GBM: the REAL result -- full shared cohort, the same 65,952/32,484 train/test split
# the DNN was meant to use.
#
# DNN: NOT the full run -- the full-scale run crashed twice (OOM), even after fixes.
# This is from the small fast-subsample sanity check (800 train / 300 test patients),
# with only 5 positive (died) cases in the entire test set. That is too few positives
# to trust as a real result -- treat it as "the pipeline works, direction is plausible",
# not as a number to compare against GBM's 0.9674 in any serious way.
gbm_auroc = 0.9674
dnn_auroc = 0.7525

gbm_notes = "REAL result -- full cohort (65,952 train / 32,484 test), shared split"
dnn_notes = "NOT full run -- small subsample only (800 train / 300 test, 5 deaths in test set). Sanity check, not comparable to GBM yet."

print(f"gbm_auroc = {gbm_auroc}  ({gbm_notes})")
print(f"dnn_auroc = {dnn_auroc}  ({dnn_notes})")


gbm_auroc = 0.9674  (REAL result -- full cohort (65,952 train / 32,484 test), shared split)
dnn_auroc = 0.7525  (NOT full run -- small subsample only (800 train / 300 test, 5 deaths in test set). Sanity check, not comparable to GBM yet.)


In [10]:
results = pd.DataFrame([
    {'model': 'NELA (demo values, not fit to INSPIRE)', 'AUROC': None, 'notes': 'fixed clinical equation, no training -- see §9'},
    {'model': 'POSSUM (demo values, not fit to INSPIRE)', 'AUROC': None, 'notes': 'fixed clinical equation, newly added -- see §9'},
    {'model': 'GBM baseline (Logistic Regression, 18 pre-op features)', 'AUROC': round(gbm_auroc, 4), 'notes': gbm_notes},
    {'model': 'DNN transformer (two-phase, 7 features)', 'AUROC': round(dnn_auroc, 4), 'notes': dnn_notes},
    {'model': 'Shickel et al. 2023 (published benchmark, 56,242 patients)', 'AUROC': 0.92, 'notes': 'the number to beat, docs/index.md §16'},
])
results


,model,AUROC,notes
0,"NELA (demo values, not fit to INSPIRE)",NaN,"fixed clinical equation, no training -- see §9"
1,"POSSUM (demo values, not fit to INSPIRE)",NaN,"fixed clinical equation, newly added -- see §9"
2,"GBM baseline (Logistic Regression, 18 pre-op f...",0.9674,"REAL result -- full cohort (65,952 train / 32,..."
3,"DNN transformer (two-phase, 7 features)",0.7525,NOT full run -- small subsample only (800 trai...
4,"Shickel et al. 2023 (published benchmark, 56,2...",0.9200,"the number to beat, docs/index.md §16"


## 13. Starter code for what's next
*(research doc §3 — new research ideas; not full implementations, but working scaffolding
to build on, using data already loaded in this notebook)*


### 13a. Organ-system feature grouping
*(research doc §2.5, §2.12 — the target architecture in your diagram)*

The grouping from `docs/index.md` §15 item 4, as a dict — this is the thing to loop over
once you split the single `TimeSeriesTransformer` into one encoder per system.


In [11]:
ORGAN_SYSTEMS = {
    'renal': {
        'labs': ['bun', 'calcium', 'chloride', 'creatinine', 'ica', 'phosphorus', 'potassium', 'sodium'],
        'ward_vitals': ['crrt', 'uo'],
    },
    'cardiovascular': {
        'labs': ['ck', 'ckmb', 'troponin_i', 'troponin_t'],
        'ward_vitals': ['hr', 'nibp_sbp', 'nibp_dbp', 'nibp_mbp', 'iabp'],
    },
    'respiratory': {
        'labs': ['be', 'hco3', 'paco2', 'pao2', 'ph', 'sao2'],
        'ward_vitals': ['fio2', 'rr', 'spo2', 'vent', 'ecmo'],
    },
    'metabolic_hepatic': {
        'labs': ['albumin', 'alp', 'alt', 'ast', 'glucose', 'hba1c', 'lacate', 'total_bilirubin', 'total_protein'],
        'ward_vitals': ['bt'],
    },
    'haematology_coagulation': {
        'labs': ['aptt', 'crp', 'd_dimer', 'fibrinogen', 'hb', 'hct', 'lymphocyte', 'platelet', 'ptinr', 'seg', 'wbc'],
        'ward_vitals': [],
    },
    'neurological': {
        'labs': [],
        'ward_vitals': ['gcs_e', 'gcs_m', 'gcs_v'],
    },
}

# Sketch: one TimeSeriesTransformer instance per system, reusing the existing pipeline
# functions unchanged -- just called once per system with that system's feature list.
# system_encoders = {}
# for system_name, feats in ORGAN_SYSTEMS.items():
#     feature_columns = feats['labs'] + feats['ward_vitals']
#     if not feature_columns:
#         continue
#     subjects_data_sys, seq_length_sys = dnn_mortality_data.load_real_subjects(
#         DATA_DIR, feature_columns, days_before_operation=5)
#     auto_dl, scaler_sys, num_feat_sys = pipeline.preprocess_for_autoencode(
#         subjects_data_sys, seq_length_sys, feature_columns)
#     system_encoders[system_name] = pipeline.train_autoencoder(auto_dl, num_feat_sys, device, epochs=10)
# -- each system_encoders[name] then produces that system's embedding (research doc §2.5 step 3);
#    fusing them (research doc §2.9) is the next piece of new code to write, not yet here.


### 13b. Attention audit scaffold
*(research doc §2.14C item 4 — genuine mechanistic evidence, not a post-hoc explanation)*

`TimeSeriesTransformer`'s encoder layers don't expose attention weights by default with
`batch_first=True` PyTorch `TransformerEncoder`. Getting them out requires either a
forward hook or rebuilding the encoder loop manually with
`need_weights=True` on each `MultiheadAttention` call — left as a TODO here rather than
silently faked, since it needs a small change to `TimeSeriesTransformer.forward()` in
`dnn_mortality_pipeline.py` (add an optional `return_attention=True` path) before it can
run. Once available, the check described in the research doc is: do attention weights
spike at the chart-times of the acute-deterioration diagnoses already found in
`docs/eda_findings.md` §4 (D65, I46, R57, J80, K72, A41)?


---

## Where to go from here

Everything above is either an existing script wired up to run end to end, or a small new
addition (the label audit, the multi-op sensitivity check, the POSSUM implementation).
The bigger pieces of new work — the system-separated architecture, the time-to-event
reframing, the Concept Bottleneck / sparse-autoencoder / prototype interpretability layers
— are design work first, code second; they're laid out in full in
`docs/research_questions_and_roadmap.md` §2.5–§2.14 and prioritised in its §4 roadmap. The
single next action, before any of that: fix the label-definition bug in Section 3 of this
notebook and re-run everything above against the full 99,886-patient cohort once it's
available.
